# A first loop

This page builds a loop of three components: a pump lifts water from a sump to the inlet of a pipe, the pipe resists the flow, and a cooler carries the water back to the sump at 40 °C. It solves the loop for its steady state, reads a pressure, a flow and a temperature out of the answer, and checks the flow against one division done by hand. [Building the Pool Reactor Loop](LOFABuildingTheLoop.ipynb) puts a full reactor model together in the same order, with more components.

In [1]:
from stream.calculations import HeatExchanger, Junction, Pump, Resistor
from stream.composition import FlowGraph, flow_edge

## Three components and three junctions

A `Pump` imposes a pressure gain of 5000 Pa along its own edge, a `Resistor` loses its resistance times the flow, here 25 000 Pa per kg/s, and a `HeatExchanger` delivers its water at 40 °C and pushes nothing. The cooler is there because every flow component carries a temperature as well as a pressure, and a loop with nothing to set the temperature leaves it undetermined. A `Junction` is a place where flow arrives and leaves. Every junction and component is named, because the names are the keys the answer is read with.

In [2]:
inlet, outlet, sump = Junction("inlet"), Junction("outlet"), Junction("sump")
pump = Pump(pressure=5e3, name="pump")
pipe = Resistor(2.5e4, name="pipe")
cooler = HeatExchanger(40.0, name="cooler")

## Edges and the flow graph

Each `flow_edge` puts a component on an edge from one junction to the next, and the direction written is the direction of positive flow. `FlowGraph` reads the edges and builds `fg.aggregator`, the system holding every equation of the loop, and `fg.kirchhoff`, the calculation holding every edge's flow. [Wire a flow network](howto/wire-a-flow-network.ipynb) covers junction names, edge direction and the sign of each gain.

In [3]:
fg = FlowGraph(
    flow_edge((sump, inlet), pump),
    flow_edge((inlet, outlet), pipe),
    flow_edge((outlet, sump), cooler),
)
agr = fg.aggregator

## Solving for the steady state

`solve_steady` finds the state where every equation holds, starting from a guess. `fg.guess_steady_state` builds that guess from a flow for each component and one temperature. `agr.save` turns the bare vector the solve returns into a state read by calculation name and variable name.

In [4]:
guess = fg.guess_steady_state({pump: 0.1, pipe: 0.1, cooler: 0.1}, 40.0)
state = agr.save(agr.solve_steady(guess))

## Reading the answer

Each component's `pressure` is its gain from inlet to outlet. A flow belongs to an edge, not to the hardware on it, so the flows sit in the Kirchhoff calculation's entry, keyed by edge; `component_edge` returns the key of the edge a component sits on.

In [5]:
flow = state[fg.kirchhoff.name][fg.kirchhoff.component_edge(pipe)]
gains = sum(state[name]["pressure"] for name in ("pump", "pipe", "cooler"))
print(f"pipe pressure gain:    {state['pipe']['pressure']:.1f} Pa")
print(f"pipe flow:             {flow:.4f} kg/s")
print(f"head / resistance:     {5e3 / 2.5e4:.4f} kg/s")
print(f"gains around the loop: {round(gains)} Pa")
print(f"cooler outlet:         {state['cooler']['Tin']:.1f} °C, given 40.0 °C")
assert abs(flow - 5e3 / 2.5e4) < 1e-9

pipe pressure gain:    -5000.0 Pa
pipe flow:             0.2000 kg/s
head / resistance:     0.2000 kg/s
gains around the loop: 0 Pa
cooler outlet:         40.0 °C, given 40.0 °C


The pipe loses about 5000 Pa, the head the pump gains, and carries about 0.2 kg/s: the head divided by the resistance, 5000 Pa divided by 25 000 Pa per kg/s. The gains around the closed loop add to zero, as the loop law requires. A component's `Tin` entry holds the temperature it hands downstream, the next component's inlet: the cooler's is the 40 °C it was given.

## Where next

The [how-to guides](howto/index.rst) take each step of this page further: wiring larger networks, choosing channels and their laws, getting a hard steady solve to converge and running transients. [Series RLC Circuit](SeriesRLCCircuit.ipynb) starts from the other end and writes the calculations themselves.